In [20]:
import torch
print(torch.__version__)

2.11.0+cu128


In [21]:
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [22]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [23]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))

1115394


In [24]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [25]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print(chars)
print("Vocabulary size:", vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocabulary size: 65


In [26]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

In [27]:
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [28]:
def encode(text):
    return [stoi[c] for c in text]

def decode(numbers):
    return ''.join(itos[i] for i in numbers)

In [29]:
# eg for enoding and decoding
sample = "Hello"

encoded = encode(sample)

print(encoded)

[20, 43, 50, 50, 53]


In [30]:
# decoding form numbers to text
decoded = decode(encoded)

print(decoded)

Hello


In [31]:
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape)

torch.Size([1115394])


In [32]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print(len(train_data))
print(len(val_data))

1003854
111540


In [33]:
# creating a context window where it helps model to train with small chunks
block_size = 8

In [34]:
# we will be feeding multiple batches simultaneously
torch.manual_seed(1337)

batch_size = 4
block_size = 8

In [35]:
def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    # Move batches to GPU/CPU
    x = x.to(device)
    y = y.to(device)

    return x, y

In [36]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

xb, yb = get_batch("train")

print(xb.shape)
print(yb.shape)

Using device: cuda
torch.Size([4, 8])
torch.Size([4, 8])


In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        # idx shape:
        # (B, T)

        logits = self.token_embedding_table(idx)

        # logits shape:
        # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [38]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

Using device: cuda


In [39]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)

print("Model created successfully!")

Model created successfully!


In [40]:
xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Input shape:", xb.shape)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Input shape: torch.Size([4, 8])
Logits shape: torch.Size([32, 65])
Loss: 4.677961826324463


In [41]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [42]:
for steps in range(5000):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if steps % 500 == 0:
        print(
            f"step {steps}: loss {loss.item():.4f}"
        )

step 0: loss 4.8857
step 500: loss 4.4507
step 1000: loss 4.0345
step 1500: loss 3.6869
step 2000: loss 3.4526
step 2500: loss 3.4358
step 3000: loss 3.1720
step 3500: loss 3.0482
step 4000: loss 3.3655
step 4500: loss 2.9445


In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Each token directly predicts the next token
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        # idx shape: (B, T)
        logits = self.token_embedding_table(idx)

        # logits shape: (B, T, vocab_size)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape

            # Cross entropy expects:
            # (B*T, C)
            logits = logits.view(B * T, C)

            # Target shape:
            # (B*T)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        # Generate one token at a time
        for _ in range(max_new_tokens):

            # Get predictions
            logits, loss = self(idx)

            # We only care about the last character
            logits = logits[:, -1, :]

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Randomly select the next token
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add the new token to our sequence
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [44]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)

In [45]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated = model.generate(
    context,
    max_new_tokens=500
)

print(
    decode(generated[0].tolist())
)


A
C;xODkRZrXwcOwd,STkO-vT,eRtK
bHiPTCkMBbUo$H:XaSGgO-33mMBc?giTaUMX:aVtLFpXezuDqhPTxv.tPVF dXE!DZaLeWQwccGwpR.fWfDGZnYu,zCkm&X
YoR&&FMtofZiEVvqbH&V!$W;KwilNZR.eg irweQsvohciK;lxW;Hmaidklv EtSXEBqqWklpD.GZ!.FNWjbO!UelJlin?yUVSu.$-vuD3 PncwKVy:pDOO!MFJg'X3UVty.OHlvBPUHIknBf&pjY-vgvIEjMk:Jmy:EJdlNBeSGklEoaW-CQoFQPdGCeRib3sr'oa-G&dM$HZLETxg$hhx&Ku,O,-LKKoAP-xTSA
mgxkXCmnvCrupD!f 'o;;3
mHh!Tm:fEtPjUy.flPUo$yj,fMFMPRD?ygKwSoB.UrHK-Nsbp!asrXeb&F&:
aadUaUWGc!dE!Dx3mLhdihVKN'Q?ya-AAnyHIrUotsvI&fn;&EZ!NMJ


In [46]:
embedding_dim = 32

token_embedding_table = nn.Embedding(
    vocab_size,
    embedding_dim
)

In [47]:
key = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
)

query = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
)

value = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
)

In [51]:
xb, yb = get_batch("train")

print("xb shape:", xb.shape)

xb shape: torch.Size([4, 8])


In [52]:
embedding_dim = 32

token_embedding_table = nn.Embedding(
    vocab_size,
    embedding_dim
).to(device)

x = token_embedding_table(xb)

print("x shape:", x.shape)

x shape: torch.Size([4, 8, 32])


In [53]:
key = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
).to(device)

query = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
).to(device)

value = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False
).to(device)

In [54]:
k = key(x)
q = query(x)
v = value(x)

print("k shape:", k.shape)
print("q shape:", q.shape)
print("v shape:", v.shape)

k shape: torch.Size([4, 8, 32])
q shape: torch.Size([4, 8, 32])
v shape: torch.Size([4, 8, 32])


In [55]:
wei = q @ k.transpose(-2, -1)

print("Attention score shape:", wei.shape)

Attention score shape: torch.Size([4, 8, 8])


In [56]:
wei = wei * (embedding_dim ** -0.5)

print(wei.shape)

torch.Size([4, 8, 8])


In [57]:
wei = wei / (embedding_dim ** 0.5)

In [58]:
tril = torch.tril(
    torch.ones(block_size, block_size, device=device)
)

print(tril)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]], device='cuda:0')


In [59]:
wei = wei.masked_fill(
    tril == 0,
    float("-inf")
)

In [60]:
wei = F.softmax(wei, dim=-1)

print(wei.shape)

torch.Size([4, 8, 8])


In [61]:
out = wei @ v

print("Attention output shape:", out.shape)

Attention output shape: torch.Size([4, 8, 32])


In [62]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(
            embedding_dim,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            embedding_dim,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            embedding_dim,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(block_size, block_size)
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        # Attention scores
        wei = q @ k.transpose(-2, -1)

        # Scale
        wei = wei * (k.shape[-1] ** -0.5)

        # Causal mask
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        # Softmax
        wei = F.softmax(wei, dim=-1)

        # Values
        v = self.value(x)

        # Attention output
        out = wei @ v

        return out

In [63]:
head_size = 16

head = Head(head_size).to(device)

out = head(x)

print("Input:", x.shape)
print("Output:", out.shape)

Input: torch.Size([4, 8, 32])
Output: torch.Size([4, 8, 16])


In [64]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            embedding_dim
        )

    def forward(self, x):

        # Run every attention head
        outputs = [
            h(x)
            for h in self.heads
        ]

        # Concatenate outputs
        out = torch.cat(
            outputs,
            dim=-1
        )

        # Project back to embedding dimension
        out = self.proj(out)

        return out

In [65]:
num_heads = 4
head_size = embedding_dim // num_heads

mha = MultiHeadAttention(
    num_heads,
    head_size
).to(device)

out = mha(x)

print("Input:", x.shape)
print("Output:", out.shape)

Input: torch.Size([4, 8, 32])
Output: torch.Size([4, 8, 32])


In [66]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                embedding_dim,
                4 * embedding_dim
            ),
            nn.GELU(),
            nn.Linear(
                4 * embedding_dim,
                embedding_dim
            )
        )

    def forward(self, x):
        return self.net(x)

In [67]:
ff = FeedForward(embedding_dim).to(device)

out = ff(x)

print("Input:", x.shape)
print("Output:", out.shape)

Input: torch.Size([4, 8, 32])
Output: torch.Size([4, 8, 32])


In [68]:
class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        head_size = embedding_dim // num_heads

        self.sa = MultiHeadAttention(
            num_heads,
            head_size
        )

        self.ffwd = FeedForward(
            embedding_dim
        )

        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )

    def forward(self, x):

        # Attention + residual connection
        x = x + self.sa(
            self.ln1(x)
        )

        # Feed-forward + residual connection
        x = x + self.ffwd(
            self.ln2(x)
        )

        return x

In [69]:
block = TransformerBlock(
    embedding_dim=32,
    num_heads=4
).to(device)

out = block(x)

print("Input:", x.shape)
print("Output:", out.shape)


Input: torch.Size([4, 8, 32])
Output: torch.Size([4, 8, 32])


In [70]:
# positional embedding
position_embedding_table = nn.Embedding(
    block_size,
    embedding_dim
).to(device)

In [71]:
positions = torch.arange(
    block_size,
    device=device
)

pos_emb = position_embedding_table(
    positions
)

print(pos_emb.shape)

torch.Size([8, 32])


In [72]:
x = token_embedding_table(xb)

x = x + pos_emb

In [73]:
# Tiny GPT
class TinyGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        block_size,
        num_heads,
        num_layers
    ):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Position embeddings
        self.position_embedding_table = nn.Embedding(
            block_size,
            embedding_dim
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads
                )
                for _ in range(num_layers)
            ]
        )

        # Final LayerNorm
        self.ln_f = nn.LayerNorm(
            embedding_dim
        )

        # Language-model head
        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )

        self.block_size = block_size

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        token_emb = self.token_embedding_table(idx)

        # Position embeddings
        positions = torch.arange(
            T,
            device=idx.device
        )

        pos_emb = self.position_embedding_table(
            positions
        )

        # Add token + position information
        x = token_emb + pos_emb

        # Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Convert embeddings → logits
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(
                B * T,
                C
            )

            targets = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens
    ):

        for _ in range(max_new_tokens):

            # Keep only the latest block_size tokens
            idx_cond = idx[:, -self.block_size:]

            # Get predictions
            logits, loss = self(idx_cond)

            # Only the last token's prediction
            logits = logits[:, -1, :]

            # Convert logits → probabilities
            probs = F.softmax(
                logits,
                dim=-1
            )

            # Sample next token
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add token to sequence
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [74]:
batch_size = 32
block_size = 64

embedding_dim = 128
num_heads = 4
num_layers = 4

In [75]:
model = TinyGPT(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    block_size=block_size,
    num_heads=num_heads,
    num_layers=num_layers
).to(device)

print("Model created!")

Model created!


In [76]:
print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

Number of parameters: 816705


In [77]:
xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Input shape:", xb.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([32, 64])
Logits shape: torch.Size([2048, 65])
Initial loss: 4.365440368652344


In [78]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [79]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in ["train", "val"]:

        losses = torch.zeros(50)

        for k in range(50):

            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()

    return results

In [80]:
max_steps = 5000

for step in range(max_steps):

    # Occasionally evaluate
    if step % 500 == 0:

        losses = estimate_loss()

        print(
            f"step {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    # Get training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear old gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

step 0: train loss 4.3683, val loss 4.3726
step 500: train loss 2.2593, val loss 2.2798
step 1000: train loss 1.9950, val loss 2.0552
step 1500: train loss 1.8349, val loss 1.9260
step 2000: train loss 1.7264, val loss 1.8752
step 2500: train loss 1.6604, val loss 1.8187
step 3000: train loss 1.6011, val loss 1.7690
step 3500: train loss 1.5675, val loss 1.7392
step 4000: train loss 1.5328, val loss 1.7121
step 4500: train loss 1.4984, val loss 1.6898


In [81]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

In [82]:
generated = model.generate(
    context,
    max_new_tokens=500
)

In [83]:
print(
    decode(
        generated[0].tolist()
    )
)


Mene to Pomembry.

HASTINGS:
Shall I save your poor between throaty?

HORTENSIO:
You well one will.

KING EDWARD IV:
But, my issue that work.

LUCIO:
Abone overnamentage!

MENENIUS:
Sir, time:
He doubtxen evil as not comClon; askness honour.

KING RICHARD II:
But a wonder who forwear so! why liegenemion.

Nurse:
I'll speaks some subst devil him.

GLOUCESTER:
With sever doom time to ounes.

QUEEN MARGARET:
These enemies themselves well be thou said;
For for commatch time wout in forcesy,
My gife 


In [84]:
prompt = "ROMEO:"

In [85]:
context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

In [86]:
generated = model.generate(
    context,
    max_new_tokens=500
)

In [87]:
print(
    decode(
        generated[0].tolist()
    )
)

ROMEO:
Then, my much softs, I have no where Maunce
The disping it doingnance weep shall die:
First things so mendly infideness of them:
How would was a down. 
HENRY BOLIINGBROKE:
Forth commanded to dame keep dot me at my brefolege
Of witness what she woes, if this still follH:
Furry! Turn brother the oftreason, bear's rance;
Made her call'd make his fail caption,
Priridenance they sick thy beguts
To cavenes to be son.

MERCUTIO:
One more headeness of my mind, speak.

LUCIO:
He yet our soul: whole, we 
